# Assignment 05 · Notebook 04: CIFAR-100, một CNN cơ bản và ba mô hình phát triển

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 4** của đề (code một CNN cơ bản và ba mô hình phát triển) trên tập
CIFAR-100 (100 lớp mịn). Bốn mô hình được huấn luyện theo đúng một cấu hình, mỗi bước chỉ thêm một cơ chế:

| | Mô hình | Cơ chế thêm vào |
|---|---|---|
| M0 | BasicCNN | gốc: [Conv → ReLU → MaxPool] × 3 → FC → FC |
| M1 | VGGNet | chồng conv 3×3, BatchNorm, GAP |
| M2 | ResNet | đường tắt $y = \mathrm{ReLU}(F(x) + x)$ |
| M3 | SE-ResNet | chú ý theo kênh $x \odot \sigma(\mathrm{MLP}(\mathrm{GAP}(x)))$ |

Kết quả ghi ra `outputs/metrics/cifar100.json`, đường học ra `outputs/figdata/cifar100_*_history.dat`,
checkpoint ra `models/cifar100_*.pt`.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import torch
from a05.data import GPUImageData
from a05.experiment import run_image_experiment
from a05.models import MODEL_LABELS
from a05.train import IMAGE_CFG

torch.backends.cudnn.benchmark = True
device = torch.device("cuda")
print("Thiết bị:", torch.cuda.get_device_name(device), "· PyTorch", torch.__version__)
print("Cấu hình:", IMAGE_CFG)

Thiết bị: NVIDIA GeForce RTX 4060 Laptop GPU · PyTorch 2.13.0+cu126
Cấu hình: {'epochs': 30, 'batch_size': 256, 'max_lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, 'grad_clip': 2.0}


## Nạp dữ liệu lên GPU

50 000 ảnh huấn luyện được tách 45 000 train / 5 000 validation (phân tầng, seed 42); 10 000 ảnh
test chính thức giữ nguyên. Cả ba nhánh nằm trên VRAM dưới dạng `uint8`.

In [2]:
data = GPUImageData("cifar100", device)
print({k: tuple(v.shape) for k, v in data.x.items()})
print("mean theo kênh:", data.mean.flatten().cpu().numpy().round(4), "std:", data.std.flatten().cpu().numpy().round(4))

{'train': (45000, 3, 32, 32), 'val': (5000, 3, 32, 32), 'test': (10000, 3, 32, 32)}
mean theo kênh: [0.5071 0.4864 0.4406] std: [0.2672 0.2563 0.276 ]


## Huấn luyện bốn mô hình

Mỗi mô hình: SGD nesterov (momentum 0,9, weight decay 5e-4), lịch OneCycle với lr cực đại 0,1,
30 epoch, lô 256, AMP fp16. Checkpoint giữ epoch có val loss nhỏ nhất.

In [3]:
results = run_image_experiment("cifar100", data)

== cifar100 / basic


  epoch  1  train 4.1680/0.0713  val 3.7879/0.1210  12.1s


  epoch  2  train 3.5609/0.1604  val 3.3568/0.1918  1.1s


  epoch  3  train 3.2314/0.2142  val 3.0717/0.2424  1.2s


  epoch  4  train 3.0030/0.2578  val 2.9066/0.2800  1.1s


...

  epoch 27  train 0.2757/0.9273  val 1.1368/0.6936  60.8s


  epoch 28  train 0.2231/0.9469  val 1.0805/0.7088  60.3s


  epoch 29  train 0.1919/0.9581  val 1.0744/0.7086  59.9s


  epoch 30  train 0.1767/0.9644  val 1.0768/0.7082  59.6s


   test acc 0.7186  top5 0.9256  macro-F1 0.7192


## Tổng hợp

In [4]:
df = pd.DataFrame(results).T
df.index = [MODEL_LABELS[n] for n in df.index]
df[["acc", "macro_f1", "top5"]] *= 100
df[["acc", "top5", "macro_f1", "params", "macs", "epoch_s", "infer_ms", "best_epoch"]].round(3)

,acc,top5,macro_f1,params,macs,epoch_s,infer_ms,best_epoch
BasicCNN,57.48,84.33,57.426,643492.0,10871808.0,2.660,0.008,30.0
VGGNet,70.71,91.90,70.786,2722084.0,379282432.0,49.179,0.093,29.0
ResNet,71.71,92.26,71.709,2764132.0,383673344.0,58.276,0.106,29.0
SE-ResNet,71.86,92.56,71.917,2786588.0,383694848.0,59.434,0.115,29.0
